### Practical No. 7: Implement Random Forest Algorithm and Evaluate the Performance

Submitted By: Shantanu Yadav            
Roll No.: 75

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("sales_data.csv")
df

,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
0,1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,North-Bob
1,1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Cash,Retail,West-Bob
2,1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.40,Returning,0.20,Bank Transfer,Retail,South-David
3,1072,2023-08-24,Bob,South,2167.94,39,Clothing,4330.03,4467.75,New,0.02,Credit Card,Retail,South-Bob
4,1061,2023-03-24,Charlie,East,3750.20,13,Electronics,637.37,692.71,New,0.08,Credit Card,Online,East-Charlie
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1010,2023-04-15,Charlie,North,4733.88,4,Food,4943.03,5442.15,Returning,0.29,Cash,Online,North-Charlie
996,1067,2023-09-07,Bob,North,4716.36,37,Clothing,1754.32,1856.40,New,0.21,Bank Transfer,Retail,North-Bob
997,1018,2023-04-27,David,South,7629.70,17,Clothing,355.72,438.27,Returning,0.06,Bank Transfer,Online,South-David
998,1100,2023-12-20,David,West,1629.47,39,Electronics,3685.03,3743.39,New,0.01,Bank Transfer,Online,West-David


In [3]:
df.columns

Index(['Product_ID', 'Sale_Date', 'Sales_Rep', 'Region', 'Sales_Amount',
       'Quantity_Sold', 'Product_Category', 'Unit_Cost', 'Unit_Price',
       'Customer_Type', 'Discount', 'Payment_Method', 'Sales_Channel',
       'Region_and_Sales_Rep'],
      dtype='object')

In [4]:
df.shape

(1000, 14)

In [5]:
df.describe()

,Product_ID,Sales_Amount,Quantity_Sold,Unit_Cost,Unit_Price,Discount
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000
mean,1050.128000,5019.265230,25.355000,2475.304550,2728.440120,0.15239
std,29.573505,2846.790126,14.159006,1417.872546,1419.399839,0.08720
min,1001.000000,100.120000,1.000000,60.280000,167.120000,0.00000
25%,1024.000000,2550.297500,13.000000,1238.380000,1509.085000,0.08000
50%,1051.000000,5019.300000,25.000000,2467.235000,2696.400000,0.15000
75%,1075.000000,7507.445000,38.000000,3702.865000,3957.970000,0.23000
max,1100.000000,9989.040000,49.000000,4995.300000,5442.150000,0.30000


In [6]:
df.isnull().sum()

Product_ID              0
Sale_Date               0
Sales_Rep               0
Region                  0
Sales_Amount            0
Quantity_Sold           0
Product_Category        0
Unit_Cost               0
Unit_Price              0
Customer_Type           0
Discount                0
Payment_Method          0
Sales_Channel           0
Region_and_Sales_Rep    0
dtype: int64

In [10]:
df.dtypes

Product_ID                int64
Sale_Date                object
Sales_Rep                object
Region                   object
Sales_Amount            float64
Quantity_Sold             int64
Product_Category         object
Unit_Cost               float64
Unit_Price              float64
Customer_Type            object
Discount                float64
Payment_Method           object
Sales_Channel            object
Region_and_Sales_Rep     object
dtype: object

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Sale_Date'] = le.fit_transform(df['Sale_Date'])
df['Sales_Rep'] = le.fit_transform(df['Sales_Rep'])
df['Region'] = le.fit_transform(df['Region'])
df['Product_Category'] = le.fit_transform(df['Product_Category'])
df['Payment_Method'] = le.fit_transform(df['Payment_Method'])
df['Sales_Channel'] = le.fit_transform(df['Sales_Channel'])
df['Region_and_Sales_Rep'] = le.fit_transform(df['Region_and_Sales_Rep'])



In [17]:
df

,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
0,1052,31,1,1,5053.97,18,3,152.75,267.22,1,0.09,1,0,6
1,1093,102,1,3,4384.02,17,3,3816.39,4209.44,1,0.11,1,1,16
2,1015,246,3,2,4631.23,30,2,261.56,371.40,1,0.20,0,1,13
3,1072,222,1,2,2167.94,39,0,4330.03,4467.75,0,0.02,2,1,11
4,1061,76,2,0,3750.20,13,1,637.37,692.71,0,0.08,2,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1010,96,2,1,4733.88,4,2,4943.03,5442.15,1,0.29,1,0,7
996,1067,234,1,1,4716.36,37,0,1754.32,1856.40,0,0.21,0,1,6
997,1018,108,3,2,7629.70,17,0,355.72,438.27,1,0.06,0,0,13
998,1100,328,3,3,1629.47,39,1,3685.03,3743.39,0,0.01,0,0,18


In [28]:
X  = df.drop('Sales_Channel',axis = 1)
y = df['Sales_Channel']

In [29]:
X

,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Region_and_Sales_Rep
0,1052,31,1,1,5053.97,18,3,152.75,267.22,1,0.09,1,6
1,1093,102,1,3,4384.02,17,3,3816.39,4209.44,1,0.11,1,16
2,1015,246,3,2,4631.23,30,2,261.56,371.40,1,0.20,0,13
3,1072,222,1,2,2167.94,39,0,4330.03,4467.75,0,0.02,2,11
4,1061,76,2,0,3750.20,13,1,637.37,692.71,0,0.08,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1010,96,2,1,4733.88,4,2,4943.03,5442.15,1,0.29,1,7
996,1067,234,1,1,4716.36,37,0,1754.32,1856.40,0,0.21,0,6
997,1018,108,3,2,7629.70,17,0,355.72,438.27,1,0.06,0,13
998,1100,328,3,3,1629.47,39,1,3685.03,3743.39,0,0.01,0,18


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [32]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42)

In [33]:
model.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
y_pred = model.predict(X_test)

In [39]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.5

Confusion Matrix:
[[46 55]
 [45 54]]

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.46      0.48       101
           1       0.50      0.55      0.52        99

    accuracy                           0.50       200
   macro avg       0.50      0.50      0.50       200
weighted avg       0.50      0.50      0.50       200

